In [ ]:
# Saliency Export & Quantification — gradient-weighted attention rollout
#
# Computes saliency maps for true-positive (TP) fundus images of each backbone × adaptation-strategy configuration
# and quantifies saliency with respect to retinal structures.
#
# Pipeline (per configuration):
#   1. Select TP images with the validation-derived Youden threshold, separately for the test set (detection) and
#      the future-incidence cohort (prediction)
#   2. Load the trained checkpoint and replace each attention module with an explicit softmax computation that keeps
#      the attention probabilities and their gradients (same computation as the original module, incl. DINOv3 RoPE
#      and LoRA); the logit before and after patching is compared to confirm this
#   3. Gradient-weighted attention rollout (Chefer, Gur & Wolf, ICCV 2021):
#        Ā_l = E_h[(∇A_l ⊙ A_l)⁺],  R = (I + Ā_L) … (I + Ā_1),  CLS row → patch-level map, min–max scaled to 0–1
#      The input requires grad, so attention in every block is gradient-weighted, including frozen blocks
#   4. Save saliency tensors and metadata; save representative heatmaps averaged over TP images of each eye
#   5. Quantify saliency within AutoMorph segmentations (artery, vein, optic disc/cup and its quadrants) and
#      summarize region-to-reference ratios with 95% CIs from an image-level bootstrap
#
# Outputs (per configuration <desc>; <tag> = test | future):
#   TENSOR_ROOT/<desc>_<tag>/       — saliency_map.pth, meta.json, tp_df.csv
#   TENSOR_ROOT/<desc>_<tag>/Mask/  — RESULTS_tp.csv (per image), ATTN_summary_<TEST|FUTURE>.csv
#   REP_ROOT/<desc>/                — Left/Right averaged heatmap and overlay PNGs
#   SUMMARY_ROOT/QC_report.csv      — TP counts, mask availability, patch and prediction checks

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # set before importing torch

import sys, math, json, random, time, warnings, gc
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from tqdm import tqdm
from sklearn.metrics import roc_curve

import matplotlib.pyplot as plt


# -------------------------------------------------------------------------
# Paths — update these to match your environment
# -------------------------------------------------------------------------
# WORK_DIR   : project root; must contain models/encoder.py with build_model
# CKPT_ROOT  : trainer.py --root_dir (per-configuration folders with ckpt.pth.tar and *_preds.csv)
# DET_CSV    : dementia_detection.csv (pngfilename, eye_orientation, ...)
# INC_CSV    : dementia_prediction.csv (pngfilename, eye_orientation, event, ...)
# OUT_BASE   : output root for saliency tensors, representative heatmaps and summary tables
# AUTOMORPH  : AutoMorph 'Results' folder (M2 artery/vein binary masks and optic disc/cup segmentations)
# -------------------------------------------------------------------------
WORK_DIR  = Path("/path/to/project/root")
CKPT_ROOT = Path("/path/to/checkpoints")
DET_CSV   = Path("/path/to/dementia_detection.csv")
INC_CSV   = Path("/path/to/dementia_prediction.csv")
OUT_BASE  = Path("/path/to/explain_out")
AUTOMORPH = Path("/path/to/AutoMorph/Results")

ARTERY_DIR  = AUTOMORPH / "M2/artery_vein/artery_binary_process"
VEIN_DIR    = AUTOMORPH / "M2/artery_vein/vein_binary_process"
ODC_RAW_DIR = AUTOMORPH / "M2/optic_disc_cup/raw"            # R channel = disc, B channel = cup

TENSOR_ROOT  = OUT_BASE / "tensors"
REP_ROOT     = OUT_BASE / "rep_heatmaps"
SUMMARY_ROOT = OUT_BASE / "summary_tables"
for _p in (TENSOR_ROOT, REP_ROOT, SUMMARY_ROOT):
    _p.mkdir(parents=True, exist_ok=True)

# 7 backbones × 3 adaptation strategies = 21 configurations (as in dementia.sh)
IMG_SIZE     = 448
ARCHES       = ["retfound", "retfound_dinov2", "openclip", "mae", "dinov2", "dinov3", "dinov3_vith16plus"]
FTS          = ["linear", "partial", "lora"]
PARTIAL_BLKS = 1          # partial: final block          → <arch>_partial_ft_1
LORA_RANK    = 4          # lora: rank 4, all blocks      → <arch>_lora_rank_4_ft_full

# model input preprocessing (same as the evaluation transform in data.py)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

N_BOOT    = 2000          # bootstrap resamples of images for the saliency ratios
BOOT_SEED = 42

PATCH_TOL_LOGIT = 1e-1    # stop if the logit changes (relative) by more than this after patching attention
PRED_WARN_DIFF  = 0.02    # warn if the probability at CAM time differs from the saved prediction by more than this
SAVE_DTYPE      = torch.float16

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
from models.encoder import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| out:", OUT_BASE)


# -------------------------------------------------------------------------
# Configuration names · Youden threshold · model loading
# -------------------------------------------------------------------------
def model_desc(arch, ft):
    if ft == "linear":  return f"{arch}_linear"
    if ft == "partial": return f"{arch}_partial_ft_{PARTIAL_BLKS}"
    if ft == "lora":    return f"{arch}_lora_rank_{LORA_RANK}_ft_full"
    raise ValueError(f"unknown ft: {ft}")


def get_youden_threshold(desc):
    """Threshold maximizing Youden's J on valid_preds.csv (as in trainer.py); falls back to res.csv."""
    vp = CKPT_ROOT / desc / "valid_preds.csv"
    if vp.exists():
        v = pd.read_csv(vp).dropna(subset=["label", "pred"])
        if v["label"].nunique() == 2:
            fpr, tpr, thr = roc_curve(v["label"], v["pred"])
            return float(thr[np.argmax(tpr - fpr)])
    rc = CKPT_ROOT / desc / "res.csv"
    if rc.exists():
        r = pd.read_csv(rc)
        if "best_threshold_valid" in r.columns and np.isfinite(r["best_threshold_valid"].iloc[0]):
            return float(r["best_threshold_valid"].iloc[0])
    return None


def load_model(arch, ft, desc):
    """Loads the state_dict saved by trainer.py; stops if any model key is missing from the checkpoint."""
    f = CKPT_ROOT / desc / "ckpt.pth.tar"
    if not f.exists():
        print(f"[SKIP] checkpoint not found: {f}")
        return None
    args = type("Args", (), {})()
    args.arch, args.ft, args.img_size, args.lora_rank = arch, ft, IMG_SIZE, LORA_RANK
    args.ft_blks = "full" if ft == "lora" else PARTIAL_BLKS
    model = build_model(args)
    sd = torch.load(f, map_location="cpu")
    if isinstance(sd, dict) and isinstance(sd.get("state_dict"), dict):
        sd = sd["state_dict"]
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:
        raise RuntimeError(f"{desc}: {len(missing)} model keys missing from the checkpoint, e.g. {missing[:5]}")
    if unexpected:
        print(f"  [WARN] {desc}: {len(unexpected)} unexpected checkpoint keys ignored, e.g. {unexpected[:5]}")
    return model.to(device).eval()


# -------------------------------------------------------------------------
# Attention capture + gradient-weighted attention rollout
#   Each block's attention is recomputed explicitly (same computation) so that the softmax probabilities and
#   their gradients can be kept; this replaces timm fused SDPA, DINOv2 memory-efficient attention and DINOv3 SDPA.
#   - DINOv3: the attention module's own apply_rope is called, so RoPE is included
#   - LoRA: attn.qkv may be a loralib MergedLinear and is called as is
# -------------------------------------------------------------------------
def find_vit_backbone(module):
    if hasattr(module, "blocks"):
        return module
    for _, child in module.named_children():
        vb = find_vit_backbone(child)
        if vb is not None:
            return vb
    return None


def _explicit_attention(attn, x, rope=None, attn_mask=None):
    """Same computation as the attention module with an explicit softmax.
    Returns (output [B,N,C], attention probabilities [B,H,N,N])."""
    B, N, _ = x.shape
    qkv = attn.qkv(x)                                   # nn.Linear / LoRA MergedLinear / DINOv3 LinearKMaskedBias
    H = attn.num_heads
    D = qkv.shape[-1] // (3 * H)
    q, k, v = qkv.reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)        # each [B,H,N,D]
    q_norm, k_norm = getattr(attn, "q_norm", None), getattr(attn, "k_norm", None)
    if isinstance(q_norm, nn.Module):
        q = q_norm(q)
    if isinstance(k_norm, nn.Module):
        k = k_norm(k)
    if rope is not None:
        if not hasattr(attn, "apply_rope"):
            raise RuntimeError(f"{type(attn).__name__}: received rope but has no apply_rope")
        q, k = attn.apply_rope(q, k, rope)              # DINOv3: (sin, cos); CLS and storage tokens are skipped
    scale = getattr(attn, "scale", None) or D ** -0.5
    logits = (q * scale) @ k.transpose(-2, -1)
    if attn_mask is not None:
        logits = logits + attn_mask
    probs = logits.softmax(dim=-1)
    drop = getattr(attn, "attn_drop", None)             # identity in eval mode
    out = (drop(probs) if isinstance(drop, nn.Module) else probs) @ v
    out = out.transpose(1, 2).reshape(B, N, H * D)
    post_norm = getattr(attn, "norm", None)             # timm scale_norm (Identity by default)
    if isinstance(post_norm, nn.Module):
        out = post_norm(out)
    out = attn.proj(out)
    pdrop = getattr(attn, "proj_drop", None)
    if isinstance(pdrop, nn.Module):
        out = pdrop(out)
    return out, probs


def patch_attention_to_capture(vit):
    """Replaces attn.forward of every block with _explicit_attention and keeps the probabilities in attn._last_attn."""
    for blk in vit.blocks:
        attn = blk.attn
        if getattr(attn, "_patched_capture", False):
            continue

        def wrapped_forward(x, *args, _attn=attn, **kwargs):
            if args:   # positional → keyword (DINOv3: attn_bias, rope / timm: attn_mask)
                names = ["attn_bias", "rope"] if hasattr(_attn, "apply_rope") else ["attn_mask"]
                for nme, val in zip(names, args):
                    kwargs.setdefault(nme, val)
            rope = kwargs.pop("rope", None)
            mask = kwargs.pop("attn_mask", None)
            bias = kwargs.pop("attn_bias", None)
            if bias is not None:
                mask = bias if mask is None else mask + bias
            out, probs = _explicit_attention(_attn, x, rope=rope, attn_mask=mask)
            _attn._last_attn = probs
            if probs.requires_grad:
                probs.retain_grad()
            return out

        attn.forward = wrapped_forward
        attn._patched_capture = True


def _logit(model, x):
    out = model(x)
    if isinstance(out, dict):
        out = out.get("logits", out.get("pred", out.get("x_norm_clstoken")))
    return out.reshape(-1)[0]


def check_patch_equivalence(model, vit, x):
    """Logit of the original model (SDPA / memory-efficient attention / RoPE) and after patching.
    Returns (original, patched); the model stays patched."""
    with torch.no_grad():
        ref = float(_logit(model, x))
    patch_attention_to_capture(vit)
    with torch.no_grad():
        new = float(_logit(model, x))
    for blk in vit.blocks:
        blk.attn._last_attn = None
    return ref, new


class GradAttnRollout:
    """CLS → patch relevance. The number of prefix tokens (CLS + register/storage tokens) is derived from the
    patch grid of the input."""

    def __init__(self, vit):
        self.vit = vit

    def _grid_and_prefix(self, T, H, W):
        ps = self.vit.patch_embed.patch_size
        ph, pw = (ps, ps) if isinstance(ps, int) else (ps[0], ps[-1])
        gh, gw = H // ph, W // pw
        if gh * gw >= T:
            raise RuntimeError(f"could not determine the patch grid (T={T}, grid={gh}x{gw})")
        return int(gh), int(gw), int(T - gh * gw)

    def compute_cam(self, model, x_norm, check_grads=False):
        """x_norm: normalized input [1,3,H,W]. Returns (cam [gh,gw] scaled to 0–1, logit)."""
        model.zero_grad(set_to_none=True)
        x = x_norm.detach().clone().requires_grad_(True)     # input grad → attention gradients in frozen blocks too
        with torch.enable_grad():
            score = _logit(model, x)
            attn_list = [blk.attn._last_attn for blk in self.vit.blocks]
            score.backward()
        grads = [A.grad for A in attn_list]
        if check_grads:
            missing = [i for i, g in enumerate(grads) if g is None]
            if missing:
                raise RuntimeError(f"blocks without attention gradients: {missing}")
        T = attn_list[0].shape[-1]
        gh, gw, prefix = self._grid_and_prefix(T, *x.shape[-2:])

        with torch.no_grad():
            v = torch.zeros(T, device=x.device, dtype=torch.float32)
            v[0] = 1.0                                         # CLS row
            # Ā_l = E_h[(∇A_l ⊙ A_l)⁺],  R = (I + Ā_L)…(I + Ā_1): propagate the CLS row from the last block down
            for A, G in zip(reversed(attn_list), reversed(grads)):
                Abar = (G[0].float() * A[0].float()).clamp(min=0).mean(dim=0)
                v = v + v @ Abar
            cam = v[prefix:].cpu().numpy().reshape(gh, gw)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-12)
        logit = float(score.detach())
        for blk in self.vit.blocks:
            blk.attn._last_attn = None
        del attn_list, grads, x, score
        return cam, logit


# -------------------------------------------------------------------------
# Metadata (eye orientation, outcome) and per-image prediction CSVs
#   test_preds.csv (pngfilename, label, pred) / prediction_preds.csv (pngfilename, event, obs_time, pred)
# -------------------------------------------------------------------------
def _stem(p):
    return Path(str(p)).stem


def _norm_eye_label(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s in ["r", "right", "rt", "od", "o.d", "1", "true"]:
        return "R"
    if s in ["l", "left", "lt", "os", "o.s", "0", "false"]:
        return "L"
    try:
        return "R" if int(float(s)) == 1 else "L"
    except Exception:
        return None


def build_maps():
    """pngfilename (stem) → eye orientation, and → event in INC_CSV."""
    eye_map, event_map = {}, {}
    for src in (DET_CSV, INC_CSV):
        if not Path(src).exists():
            print(f"  [WARN] not found: {src}")
            continue
        cols = pd.read_csv(src, nrows=0).columns
        use = [c for c in ("pngfilename", "eye_orientation", "event") if c in cols]
        d = pd.read_csv(src, usecols=use, low_memory=False)
        stems = d["pngfilename"].map(_stem)
        if "eye_orientation" in d:
            for s_, e_ in zip(stems, d["eye_orientation"]):
                code_ = _norm_eye_label(e_)
                if code_:
                    eye_map.setdefault(s_, code_)
        if src == INC_CSV and "event" in d:
            event_map = dict(zip(stems, pd.to_numeric(d["event"], errors="coerce")))
    print(f"[maps] eye {len(eye_map):,} | event ({Path(INC_CSV).name}) {len(event_map):,}")
    if not eye_map:
        print("  [WARN] no eye_orientation → temporal/nasal are NaN and one 'All' heatmap is saved")
    return eye_map, event_map


def load_pred_csvs(desc, eye_map, event_map):
    d = CKPT_ROOT / desc
    tp, fp = d / "test_preds.csv", d / "prediction_preds.csv"
    if not tp.exists() or not fp.exists():
        print(f"[SKIP] prediction CSVs not found: {d}")
        return None, None
    test = pd.read_csv(tp).dropna(subset=["pred"]).reset_index(drop=True)
    fut  = pd.read_csv(fp).dropna(subset=["pred"]).reset_index(drop=True)
    if event_map:          # outcome from the cohort file, so TP selection follows its event definition
        new_ev = fut["pngfilename"].map(_stem).map(event_map)
        n_chg = int((new_ev.notna() & (new_ev != fut["event"])).sum())
        if n_chg:
            print(f"  · event updated from {Path(INC_CSV).name} for {n_chg:,} rows")
        fut["event"] = new_ev.fillna(fut["event"])
    for df in (test, fut):
        df["eye_orientation"] = df["pngfilename"].map(_stem).map(eye_map)
    return test, fut


# -------------------------------------------------------------------------
# Saliency export per TP image (saliency_map.pth / meta.json / tp_df.csv)
#   - model input: Resize → ToTensor → Normalize (as in data.py)
#   - tp_df.csv holds only the exported rows, aligned 1:1 with the saliency tensor
# -------------------------------------------------------------------------
to01      = Compose([Resize((IMG_SIZE, IMG_SIZE)), ToTensor()])
normalize = Normalize(IMAGENET_MEAN, IMAGENET_STD)


def resolve_img_path(p):
    p = str(p)
    return p if os.path.isabs(p) else str(WORK_DIR / p)


def load_inputs(img_path):
    x01 = to01(Image.open(img_path).convert("RGB"))          # [3,H,W] in 0–1 (overlay background)
    return x01, normalize(x01).unsqueeze(0)                   # model input [1,3,H,W]


def _none_if_nan(v):
    return None if (v is None or (isinstance(v, float) and np.isnan(v))) else v


def export_tp_pack(tp_df, out_dir, rollout, model):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cams, metas, kept, diffs = [], [], [], []
    for i, row in tqdm(tp_df.iterrows(), total=len(tp_df), desc=f"Export {out_dir.name}"):
        path = resolve_img_path(row["pngfilename"])
        if not os.path.exists(path):
            continue
        _, x = load_inputs(path)
        cam_g, logit = rollout.compute_cam(model, x.to(device), check_grads=(len(kept) == 0))
        cam = cv2.resize(cam_g.astype(np.float32), (IMG_SIZE, IMG_SIZE),
                         interpolation=cv2.INTER_CUBIC).clip(0.0, 1.0)
        p_cam = 1.0 / (1.0 + math.exp(-logit))
        diffs.append(abs(p_cam - float(row["pred"])))
        cams.append(torch.from_numpy(cam).to(SAVE_DTYPE)[None, None])
        kept.append(i)
        metas.append({
            "pngfilename": str(row["pngfilename"]),
            "eye_orientation": _none_if_nan(row.get("eye_orientation")),
            "pred": float(row["pred"]),
            "pred_cam": float(p_cam),
            "label_or_event": int(row["label"]) if "label" in row else int(row.get("event", -1)),
        })
    if not kept:
        print("No TP images to export:", out_dir)
        (out_dir / "NO_TP").write_text("no true-positive images\n")      # marks the configuration as done
        return False, {"n_tp": int(len(tp_df)), "n_exported": 0}
    (out_dir / "NO_TP").unlink(missing_ok=True)
    torch.save(torch.cat(cams, 0).contiguous(), out_dir / "saliency_map.pth")
    with open(out_dir / "meta.json", "w") as f:
        json.dump(metas, f, indent=1, ensure_ascii=False, default=str)
    tp = tp_df.loc[kept].reset_index(drop=True)
    tp["pred_cam"] = [m["pred_cam"] for m in metas]
    tp.to_csv(out_dir / "tp_df.csv", index=False)
    d = np.asarray(diffs)
    qc = {"n_tp": int(len(tp_df)), "n_exported": int(len(kept)),
          "pred_diff_median": float(np.median(d)), "pred_diff_max": float(d.max())}
    warn = "" if qc["pred_diff_max"] <= PRED_WARN_DIFF else "  [WARN] large difference — check preprocessing/weights"
    print(f"[✓] {out_dir.name}: N={len(kept)} | |p(CAM) − saved pred| median "
          f"{qc['pred_diff_median']:.4f}, max {qc['pred_diff_max']:.4f}{warn}")
    return True, qc


# -------------------------------------------------------------------------
# Representative heatmaps — TP maps averaged per eye (L / R), FOV-masked and renormalized to 0–1 within the FOV
# -------------------------------------------------------------------------
def make_fov_mask(h=IMG_SIZE, w=IMG_SIZE):
    yy, xx = np.ogrid[:h, :w]
    cy, cx, r = h // 2, w // 2, min(h, w) // 2
    return (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2

FOV = make_fov_mask()


def _load_pack(src_dir):
    p = Path(src_dir)
    cam = torch.load(p / "saliency_map.pth", map_location="cpu").float()
    with open(p / "meta.json") as f:
        metas = json.load(f)
    return cam, metas


def _background(meta):
    """Overlay background: the original image at the model input size (0–1)."""
    x01, _ = load_inputs(resolve_img_path(meta["pngfilename"]))
    return x01.permute(1, 2, 0).numpy()


def _avg_heatmap(cam, indices):
    if not indices:
        return None
    mean = cam[indices, 0].numpy().mean(axis=0) * FOV
    mn, mx = mean[FOV].min(), mean[FOV].max()
    if mx > mn:
        mean = (mean - mn) / (mx - mn + 1e-8)
    return mean


def _overlay_cam(rgb_hw3, cam_hw, alpha=0.45, cmap_name="jet"):
    cam_rgb = plt.get_cmap(cmap_name)(cam_hw)[..., :3]
    return np.clip((1 - alpha) * rgb_hw3 + alpha * cam_rgb, 0, 1)


def save_heatmap_and_overlay(src_dir, desc, set_name="TEST", seed=0, cmap="jet"):
    cam, metas = _load_pack(src_dir)
    groups = {"Left":  [i for i, m in enumerate(metas) if _norm_eye_label(m.get("eye_orientation")) == "L"],
              "Right": [i for i, m in enumerate(metas) if _norm_eye_label(m.get("eye_orientation")) == "R"]}
    if not groups["Left"] and not groups["Right"]:
        groups = {"All": list(range(len(metas)))}
    out_dir = REP_ROOT / desc
    out_dir.mkdir(parents=True, exist_ok=True)
    for gi, (name, idxs) in enumerate(groups.items()):
        avg = _avg_heatmap(cam, idxs)
        if avg is None:
            continue
        plt.imsave(out_dir / f"{set_name}_{name}_heatmap.png", avg, cmap=cmap)
        ri = random.Random(seed + gi).choice(idxs)
        plt.imsave(out_dir / f"{set_name}_{name}_overlay.png", _overlay_cam(_background(metas[ri]), avg, 0.45, cmap))
        print(f"    heatmap/overlay {set_name}_{name} (n={len(idxs)})")


# -------------------------------------------------------------------------
# AutoMorph masks and regional saliency per image
#   vessels (artery / vein), optic disc (disc ∪ cup) and its quadrants (two diagonals through the centroid;
#   temporal / nasal assigned by laterality). Each map is normalized to sum to 100 within the circular FOV; the mean
#   saliency per pixel is computed for each region (reference region = FOV outside vessels and optic disc).
# -------------------------------------------------------------------------
_MASK_EXTS  = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
_MASK_KEYS  = ("artery", "vein", "odc")
_MASK_CACHE = {}          # stem → bit-packed masks, so each AutoMorph file is read once per session


def _find(root, stem):
    for ext in _MASK_EXTS:
        p = root / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def _to_bool(img):
    m = np.array(img.resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST))
    if m.ndim == 3:
        m = m.max(axis=-1)
    return m > 0


def _read_masks(stem):
    pa, pv, po = _find(ARTERY_DIR, stem), _find(VEIN_DIR, stem), _find(ODC_RAW_DIR, stem)
    artery = _to_bool(Image.open(pa)) if pa else None
    vein   = _to_bool(Image.open(pv)) if pv else None
    odc = None
    if po:
        try:
            arr = np.array(Image.open(po).convert("RGB"))
            rb = ((arr[..., 0] > 0) | (arr[..., 2] > 0)).astype(np.uint8) * 255      # R = disc, B = cup
            odc = _to_bool(Image.fromarray(rb))
        except Exception:
            odc = None
    return artery, vein, odc


def get_masks(pngfilename):
    """(artery, vein, odc) — each a bool [H,W] array or None."""
    stem = _stem(pngfilename)
    if stem not in _MASK_CACHE:
        _MASK_CACHE[stem] = {k: (np.packbits(m) if m is not None else None)
                             for k, m in zip(_MASK_KEYS, _read_masks(stem))}
    n = IMG_SIZE * IMG_SIZE
    return tuple(None if _MASK_CACHE[stem][k] is None
                 else np.unpackbits(_MASK_CACHE[stem][k], count=n).reshape(IMG_SIZE, IMG_SIZE).astype(bool)
                 for k in _MASK_KEYS)


def split_odc_quadrants(odc):
    """Splits the optic disc mask into four quadrants by the two diagonals through its centroid."""
    ys, xs = np.nonzero(odc)
    if ys.size == 0:
        return None
    cy, cx = ys.mean(), xs.mean()
    yy, xx = np.mgrid[0:odc.shape[0], 0:odc.shape[1]]
    s1 = (yy - cy) - (xx - cx)
    s2 = (yy - cy) + (xx - cx)
    return {"superior": (s1 < 0) & (s2 < 0) & odc, "inferior": (s1 > 0) & (s2 > 0) & odc,
            "right":    (s1 < 0) & (s2 > 0) & odc, "left":     (s1 > 0) & (s2 < 0) & odc}


def _rmean(gc, m):
    n = int(m.sum())
    return float(gc[m].sum() / n) if n else float("nan")


def region_record(cam, artery, vein, odc, eye_code):
    gc = cam.astype(np.float64) * FOV
    s = gc.sum()
    if s <= 0:
        return None
    gc = 100.0 * gc / s                                     # sum = 100 within the FOV
    a, v = artery & FOV, vein & FOV
    both = a | v
    rec = {"veins_n": _rmean(gc, v), "not_veins_n": _rmean(gc, FOV & ~v),
           "arteries_n": _rmean(gc, a), "not_arteries_n": _rmean(gc, FOV & ~a),
           "both_n": _rmean(gc, both), "not_both_n": _rmean(gc, FOV & ~both),
           "optic_disc_n": np.nan, "not_optic_disc_n": np.nan, "not_optic_disc_not_both_n": np.nan,
           "optic_disc_temporal_n": np.nan, "optic_disc_nasal_n": np.nan,
           "optic_disc_superior_n": np.nan, "optic_disc_inferior_n": np.nan}
    if odc is not None:
        o = odc & FOV
        rec["optic_disc_n"] = _rmean(gc, o)
        rec["not_optic_disc_n"] = _rmean(gc, FOV & ~o)
        rec["not_optic_disc_not_both_n"] = _rmean(gc, FOV & ~(both | o))
        q = split_odc_quadrants(o)
        if q is not None:
            rec["optic_disc_superior_n"] = _rmean(gc, q["superior"])
            rec["optic_disc_inferior_n"] = _rmean(gc, q["inferior"])
            if eye_code in ("L", "R"):                      # right eye: temporal on the image left; left eye: right
                tmp, nas = (q["left"], q["right"]) if eye_code == "R" else (q["right"], q["left"])
                rec["optic_disc_temporal_n"] = _rmean(gc, tmp)
                rec["optic_disc_nasal_n"] = _rmean(gc, nas)
    return rec


def compute_metrics_for_dataset(pack_dir):
    """Regional saliency of each TP image in a pack → Mask/RESULTS_tp.csv. Returns (results, mask counts)."""
    pack_dir = Path(pack_dir)
    tp = pd.read_csv(pack_dir / "tp_df.csv")
    cams = torch.load(pack_dir / "saliency_map.pth", map_location="cpu").float().numpy()[:, 0]
    if len(tp) != len(cams):
        raise RuntimeError(f"{pack_dir.name}: tp_df ({len(tp)}) and saliency ({len(cams)}) lengths differ")
    recs, n_vessel = [], 0
    for i in tqdm(range(len(tp)), desc=f"Masks {pack_dir.name}"):
        row = tp.iloc[i]
        artery, vein, odc = get_masks(row["pngfilename"])
        if artery is None or vein is None:                  # no AutoMorph segmentation → excluded
            continue
        n_vessel += 1
        rec = region_record(cams[i], artery, vein, odc, _norm_eye_label(row.get("eye_orientation")))
        if rec is None:
            continue
        rec.update(pngfilename=row["pngfilename"], eye_orientation=row.get("eye_orientation"),
                   has_optic_disc=odc is not None)
        recs.append(rec)
    res = pd.DataFrame(recs)
    (pack_dir / "Mask").mkdir(exist_ok=True)
    res.to_csv(pack_dir / "Mask" / "RESULTS_tp.csv", index=False)
    n_od = int(res["has_optic_disc"].sum()) if len(res) else 0
    print(f"    masks: TP {len(tp)} → with vessel masks {n_vessel} → with optic disc mask {n_od}")
    return res, {"n_with_vessel_masks": int(n_vessel), "n_with_od_masks": n_od}


# -------------------------------------------------------------------------
# Attention-ratio summary with image-level bootstrap 95% CIs
#   ratio = mean saliency per pixel in a region / that in the reference region, computed per image, then averaged
# -------------------------------------------------------------------------
RATIO_SPECS = [
    ("vascular_attention_vein",       "veins_n"),
    ("vascular_attention_artery",     "arteries_n"),
    ("vascular_attention_both",       "both_n"),
    ("optic_disc_attention",          "optic_disc_n"),
    ("optic_disc_attention_temporal", "optic_disc_temporal_n"),
    ("optic_disc_attention_nasal",    "optic_disc_nasal_n"),
    ("optic_disc_attention_superior", "optic_disc_superior_n"),
    ("optic_disc_attention_inferior", "optic_disc_inferior_n"),
]


def _safe_ratio(num, den):
    num = pd.to_numeric(num, errors="coerce").to_numpy(float)
    den = pd.to_numeric(den, errors="coerce").to_numpy(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = num / den
    r[~np.isfinite(den) | (den <= 0)] = np.nan
    return r


def bootstrap_mean_ci(values, n_boot=N_BOOT, seed=BOOT_SEED):
    """Mean over images and percentile 95% CI from resampling images with replacement.
    Returns (mean, 2.5%, 97.5%, number of images)."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.nan, np.nan, np.nan, 0
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for s0 in range(0, n_boot, 250):                       # in chunks to limit memory
        b = min(250, n_boot - s0)
        means[s0:s0 + b] = v[rng.integers(0, v.size, size=(b, v.size))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(v.mean()), float(lo), float(hi), int(v.size)


def summarize_attention(res_df, label=""):
    ref = res_df["not_optic_disc_not_both_n"]
    rows = {}
    for name, col in RATIO_SPECS:
        m, lo, hi, n = bootstrap_mean_ci(_safe_ratio(res_df[col], ref))
        rows[name] = {"mean": m, "95%_CI_low": lo, "95%_CI_high": hi, "n_used": n}
    out = pd.DataFrame(rows).T
    print(f"    [{label}] " + " | ".join(f"{nm.replace('optic_disc_attention', 'OD').replace('vascular_attention_', 'V-')} "
                                         f"{out.loc[nm, 'mean']:.2f}" for nm, _ in RATIO_SPECS))
    return out

In [ ]:
# Main loop — iterates over all 21 configurations and runs the full pipeline:
#   TP selection → load checkpoint → patch check → saliency export → representative heatmaps
#   → regional quantification against AutoMorph masks → attention-ratio summary
#
# Controls:
#   SKIP_DESCS : configuration names to skip (e.g. {"dinov3_vith16plus_linear"})
#   RESUME     : skip configurations whose summaries already exist (re-running continues after an interruption)
# QC per configuration is written to SUMMARY_ROOT/QC_report.csv.


def free_gpu(vit=None):
    if vit is not None:
        for blk in vit.blocks:
            if hasattr(blk.attn, "_last_attn"):
                blk.attn._last_attn = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _done(desc):
    return all((TENSOR_ROOT / f"{desc}_{tag}" / "Mask" / f"ATTN_summary_{kind}.csv").exists()
               or (TENSOR_ROOT / f"{desc}_{tag}" / "NO_TP").exists()
               for tag, kind in (("test", "TEST"), ("future", "FUTURE")))


eye_map, event_map = build_maps()

SKIP_DESCS = set()
RESUME     = True
QC_PATH    = SUMMARY_ROOT / "QC_report.csv"
QC_ROWS    = pd.read_csv(QC_PATH).to_dict("records") if (RESUME and QC_PATH.exists()) else []

for ft in FTS:
    for arch in ARCHES:
        desc = model_desc(arch, ft)
        if desc in SKIP_DESCS or (RESUME and _done(desc)):
            print(f"[SKIP] {desc}")
            continue
        model = vit = rollout = None
        t0 = time.time()
        try:
            thr = get_youden_threshold(desc)
            if thr is None:
                print(f"[SKIP] no threshold: {desc}")
                continue
            test_df, fut_df = load_pred_csvs(desc, eye_map, event_map)
            if test_df is None:
                continue
            # TP images: predicted positive at the validation Youden threshold and dementia-positive / incident case
            test_tp = test_df[(test_df["pred"] >= thr) & (test_df["label"] == 1)].reset_index(drop=True)
            fut_tp  = fut_df[(fut_df["pred"] >= thr) & (fut_df["event"] == 1)].reset_index(drop=True)
            print("\n" + "=" * 88)
            print(f"[RUN] {desc} | thr={thr:.6f} | TP — TEST {len(test_tp)} | FUTURE {len(fut_tp)}")
            print("=" * 88)
            if not len(test_tp) and not len(fut_tp):
                for tag in ("test", "future"):
                    (TENSOR_ROOT / f"{desc}_{tag}").mkdir(parents=True, exist_ok=True)
                    (TENSOR_ROOT / f"{desc}_{tag}" / "NO_TP").write_text("no true-positive images\n")
                print(f"[SKIP] no TP images: {desc}")
                continue

            model = load_model(arch, ft, desc)
            if model is None:
                continue
            vit = find_vit_backbone(model)
            if vit is None:
                raise RuntimeError("no ViT backbone with .blocks found")

            # logit before vs after patching (RoPE, LoRA and fused attention included)
            first = test_tp if len(test_tp) else fut_tp
            _, x_chk = load_inputs(resolve_img_path(first["pngfilename"].iloc[0]))
            ref_logit, new_logit = check_patch_equivalence(model, vit, x_chk.to(device))
            dlogit = abs(ref_logit - new_logit)
            print(f"  · patch check: logit {ref_logit:.5f} (original) / {new_logit:.5f} (patched), diff {dlogit:.1e}")
            if dlogit > PATCH_TOL_LOGIT * max(1.0, abs(ref_logit)):
                raise RuntimeError(f"patched attention differs from the original model (diff {dlogit:.3e})")
            rollout = GradAttnRollout(vit)

            qc = {"desc": desc, "arch": arch, "ft": ft, "threshold": thr, "patch_logit_diff": dlogit}
            for tag, tp_df_, kind, seed in (("test", test_tp, "TEST", 123), ("future", fut_tp, "FUTURE", 456)):
                out_dir = TENSOR_ROOT / f"{desc}_{tag}"
                ok, eq = export_tp_pack(tp_df_, out_dir, rollout, model)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                qc.update({f"{tag}_{k}": v for k, v in eq.items()})
                if not ok:
                    continue
                save_heatmap_and_overlay(out_dir, desc, set_name=kind, seed=seed)
                res, mq = compute_metrics_for_dataset(out_dir)
                qc.update({f"{tag}_{k}": v for k, v in mq.items()})
                if len(res):
                    summarize_attention(res, label=f"{kind}:{desc}").to_csv(out_dir / "Mask" / f"ATTN_summary_{kind}.csv")
            qc["minutes"] = round((time.time() - t0) / 60, 1)
            QC_ROWS = [r for r in QC_ROWS if r.get("desc") != desc] + [qc]
            pd.DataFrame(QC_ROWS).to_csv(QC_PATH, index=False)

        except torch.cuda.OutOfMemoryError as e:
            print(f"[OOM] {desc} — restart the kernel and run this configuration again. ({e})")
        except Exception as e:
            print(f"[ERROR] {desc}: {type(e).__name__}: {e}")
        finally:
            free_gpu(vit)
            del model, vit, rollout
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print("\nDONE")
if QC_PATH.exists():
    display(pd.read_csv(QC_PATH))

In [ ]:
# Performance summarizer — detection (Table 2) and future incidence (Table 3)
#
# Reads the per-image predictions saved by trainer.py for the 21 configurations and computes:
#   - dementia detection (test set): AUROC, and sensitivity / specificity at the validation-set Youden threshold
#   - future dementia incidence: Harrell's C-index with the model output as a risk score
#   - 95% CIs from 2,000 bootstrap resamples of images (the same resamples for every configuration)
#   - adjusted OR (logistic regression) and HR (Cox model) per 0.1-unit increase in model output (output × 10),
#     adjusted for the six CAIDE components (age, sex, systolic BP, BMI, total cholesterol, physical activity);
#     Wald CIs; images with missing covariates are excluded
#   - CAIDE score without education as a clinical comparator (images with all six components)
#
# Inputs : CKPT_ROOT/<desc>/valid_preds.csv, test_preds.csv, prediction_preds.csv; DET_CSV; INC_CSV
#          (event and obs_time are taken from INC_CSV)
# Outputs: SUMMARY_ROOT/Table2_detection.csv, Table3_prediction.csv, performance_summary_raw.csv
# The C-index bootstrap is the slowest step (74k images × 22 scores × 2,000 resamples); N_JOBS sets the CPU workers.

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, roc_curve
import statsmodels.api as sm
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")


# -------------------------------------------------------------------------
# Paths — update to match your environment
# -------------------------------------------------------------------------
CKPT_ROOT    = Path("/path/to/checkpoints")
DET_CSV      = Path("/path/to/dementia_detection.csv")
INC_CSV      = Path("/path/to/dementia_prediction.csv")
SUMMARY_ROOT = Path("/path/to/explain_out/summary_tables")
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

N_BOOT = 2000
SEED   = 42
N_JOBS = -1               # workers for the C-index bootstrap (-1 = all CPU cores)

ARCH_ORDER  = ["retfound", "retfound_dinov2", "openclip", "mae", "dinov2", "dinov3", "dinov3_vith16plus"]
FT_ORDER    = ["linear", "partial", "lora"]
ARCH_PRETTY = {"retfound": "RETFound-MAE", "retfound_dinov2": "RETFound-DINOv2", "openclip": "OpenCLIP",
               "mae": "MAE", "dinov2": "DINOv2", "dinov3": "DINOv3-L", "dinov3_vith16plus": "DINOv3-H+"}
DOMAIN      = {a: ("Retina" if a.startswith("retfound") else "Natural") for a in ARCH_ORDER}
FT_PRETTY   = {"linear": "Linear probing", "partial": "Partial fine-tuning", "lora": "LoRA"}
COVARS      = ["STDY_AGE", "SEXINT", "cholesterol_updated", "sbp", "bmi", "EXERCISE_STATUS_bin"]
CAIDE_COMPONENTS = ["STDY_AGE", "SEXINT", "sbp", "bmi", "cholesterol_updated", "EXERCISE_STATUS"]


def model_desc(arch, ft):
    if ft == "linear":  return f"{arch}_linear"
    if ft == "partial": return f"{arch}_partial_ft_1"
    return f"{arch}_lora_rank_4_ft_full"


# -------------------------------------------------------------------------
# Data and predictions
# -------------------------------------------------------------------------
def add_covariates(df):
    out = df.copy()
    if "SEXINT" not in out.columns:
        out["SEXINT"] = (out["SEX"] == "M").astype(int)
    ex = pd.to_numeric(out["EXERCISE_STATUS"], errors="coerce")
    out["EXERCISE_STATUS_bin"] = np.where(ex >= 2, 1.0, np.where(ex <= 1, 0.0, np.nan))   # regular physical activity
    for c in ["STDY_AGE", "SEXINT", "cholesterol_updated", "sbp", "bmi", "CAIDE_noAPOE"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def read_preds(desc, fname, ref):
    """Per-image predictions aligned with the rows of `ref` (trainer.py saves them in dataset order)."""
    f = CKPT_ROOT / desc / fname
    if not f.exists():
        return None
    d = pd.read_csv(f)
    same_order = (len(d) == len(ref) and
                  (d["pngfilename"].astype(str).to_numpy() == ref["pngfilename"].astype(str).to_numpy()).all())
    if not same_order:
        d = ref[["pngfilename"]].merge(d[["pngfilename", "pred"]], on="pngfilename", how="left", validate="one_to_one")
    return d["pred"].to_numpy(float)


def youden_threshold(desc):
    v = pd.read_csv(CKPT_ROOT / desc / "valid_preds.csv").dropna(subset=["label", "pred"])
    fpr, tpr, thr = roc_curve(v["label"].astype(int), v["pred"])
    return float(thr[np.argmax(tpr - fpr)])


det = pd.read_csv(DET_CSV, low_memory=False)
det_test = add_covariates(det[det["test"] == True].reset_index(drop=True))
inc = add_covariates(pd.read_csv(INC_CSV, low_memory=False).reset_index(drop=True))

y_test = det_test["label"].to_numpy(int)
t_inc  = pd.to_numeric(inc["obs_time"]).to_numpy(float)
e_inc  = pd.to_numeric(inc["event"]).to_numpy(int)

configs, S_test, R_inc, thresholds = [], [], [], []
for arch in ARCH_ORDER:
    for ft in FT_ORDER:
        desc = model_desc(arch, ft)
        s = read_preds(desc, "test_preds.csv", det_test)
        r = read_preds(desc, "prediction_preds.csv", inc)
        if s is None or r is None or not (CKPT_ROOT / desc / "valid_preds.csv").exists():
            print(f"[SKIP] predictions not found: {desc}")
            continue
        configs.append((arch, ft, desc))
        S_test.append(s)
        R_inc.append(r)
        thresholds.append(youden_threshold(desc))
S_test, R_inc, thresholds = np.column_stack(S_test), np.column_stack(R_inc), np.asarray(thresholds)
M = len(configs)

# CAIDE comparator: images with all six components
caide_t_ok = det_test[CAIDE_COMPONENTS].notna().all(axis=1).to_numpy()
caide_i_ok = inc[CAIDE_COMPONENTS].notna().all(axis=1).to_numpy()
y_c, s_c = y_test[caide_t_ok], det_test["CAIDE_noAPOE"].to_numpy(float)[caide_t_ok]
t_c, e_c, r_c = t_inc[caide_i_ok], e_inc[caide_i_ok], inc["CAIDE_noAPOE"].to_numpy(float)[caide_i_ok]

print(f"test set {len(y_test):,} images ({y_test.sum():,} dementia) | future-incidence cohort {len(e_inc):,} images "
      f"({e_inc.sum():,} incident) | {M} configurations")


# -------------------------------------------------------------------------
# Metrics
# -------------------------------------------------------------------------
def boot_idx(n, b):
    """Indices of bootstrap resample b (resampling images with replacement); identical for all configurations."""
    return np.random.default_rng([SEED, b]).integers(0, n, size=n)


def sens_spec(y, s, thr):
    pos = s >= thr
    return float((pos & (y == 1)).sum() / (y == 1).sum()), float((~pos & (y == 0)).sum() / (y == 0).sum())


def cindex(t, risk, e):
    return concordance_index(event_times=t, predicted_scores=-risk, event_observed=e)   # higher risk → shorter time


def adjusted_or(y, pred, X):
    m = X.notna().all(axis=1).to_numpy() & np.isfinite(pred)
    Xd = sm.add_constant(pd.concat([pd.Series(pred[m] * 10.0, index=X.index[m], name="pred10"), X[m]], axis=1))
    fit = sm.Logit(pd.Series(y[m], index=X.index[m]), Xd).fit(disp=False)
    lo, hi = fit.conf_int().loc["pred10"]
    return np.exp(fit.params["pred10"]), np.exp(lo), np.exp(hi), float(fit.pvalues["pred10"]), int(m.sum())


def adjusted_hr(t, e, pred, X):
    m = X.notna().all(axis=1).to_numpy() & np.isfinite(pred) & np.isfinite(t)
    d = pd.concat([pd.DataFrame({"obs_time": t[m], "event": e[m], "pred10": pred[m] * 10.0}, index=X.index[m]),
                   X[m]], axis=1)
    cph = CoxPHFitter().fit(d, duration_col="obs_time", event_col="event")
    lo, hi = cph.confidence_intervals_.loc["pred10"]
    return (np.exp(cph.params_["pred10"]), np.exp(lo), np.exp(hi), float(cph.summary.loc["pred10", "p"]),
            int(m.sum()), int(e[m].sum()))


def _cindex_chunk(bs, t, e, R, tc, ec, rc):
    out = []
    for b in bs:
        i, ic = boot_idx(len(t), b), boot_idx(len(tc), b)
        out.append([cindex(t[i], R[i, j], e[i]) for j in range(R.shape[1])] + [cindex(tc[ic], rc[ic], ec[ic])])
    return out


def ci(a):
    return np.nanpercentile(a, [2.5, 97.5], axis=0)


# point estimates
auc  = np.array([roc_auc_score(y_test, S_test[:, j]) for j in range(M)])
ss   = np.array([sens_spec(y_test, S_test[:, j], thresholds[j]) for j in range(M)])
cidx = np.array([cindex(t_inc, R_inc[:, j], e_inc) for j in range(M)])
auc_caide, cidx_caide = roc_auc_score(y_c, s_c), cindex(t_c, r_c, e_c)

# bootstrap — detection (test set)
auc_b, sens_b, spec_b = (np.full((N_BOOT, M), np.nan) for _ in range(3))
auc_caide_b = np.full(N_BOOT, np.nan)
for b in tqdm(range(N_BOOT), desc="bootstrap AUROC"):
    i, ic = boot_idx(len(y_test), b), boot_idx(len(y_c), b)
    yb = y_test[i]
    if yb.min() == yb.max():                               # single-class resample (not expected at this size)
        continue
    for j in range(M):
        auc_b[b, j] = roc_auc_score(yb, S_test[i, j])
        sens_b[b, j], spec_b[b, j] = sens_spec(yb, S_test[i, j], thresholds[j])
    if y_c[ic].min() < y_c[ic].max():
        auc_caide_b[b] = roc_auc_score(y_c[ic], s_c[ic])

# bootstrap — future incidence (parallel over chunks of resamples)
chunks = np.array_split(np.arange(N_BOOT), max(1, N_BOOT // 50))
res = Parallel(n_jobs=N_JOBS)(delayed(_cindex_chunk)(bs, t_inc, e_inc, R_inc, t_c, e_c, r_c)
                              for bs in tqdm(chunks, desc="bootstrap C-index"))
cidx_all_b = np.array([row for chunk in res for row in chunk])
cidx_b, cidx_caide_b = cidx_all_b[:, :M], cidx_all_b[:, M]

# adjusted associations
X_test, X_inc = det_test[COVARS], inc[COVARS]
ors = [adjusted_or(y_test, S_test[:, j], X_test) for j in range(M)]
hrs = [adjusted_hr(t_inc, e_inc, R_inc[:, j], X_inc) for j in range(M)]


# -------------------------------------------------------------------------
# Tables
# -------------------------------------------------------------------------
def fmt(m, lo, hi):
    return f"{m:.3f} ({lo:.3f}–{hi:.3f})" if np.isfinite(m) else ""


def fmt_p(p):
    return "" if not np.isfinite(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")


(auc_lo, auc_hi), (sens_lo, sens_hi), (spec_lo, spec_hi), (c_lo, c_hi) = ci(auc_b), ci(sens_b), ci(spec_b), ci(cidx_b)
tab2, tab3, raw = [], [], []
for j, (arch, ft, desc) in enumerate(configs):
    base = {"Model": ARCH_PRETTY[arch], "Pretraining domain": DOMAIN[arch], "Adaptation strategy": FT_PRETTY[ft]}
    o, h = ors[j], hrs[j]
    tab2.append({**base,
                 "AUROC (95% CI)": fmt(auc[j], auc_lo[j], auc_hi[j]),
                 "Sensitivity (95% CI)": fmt(ss[j, 0], sens_lo[j], sens_hi[j]),
                 "Specificity (95% CI)": fmt(ss[j, 1], spec_lo[j], spec_hi[j]),
                 "Adjusted OR (95% CI)": fmt(o[0], o[1], o[2]), "P value": fmt_p(o[3])})
    tab3.append({**base,
                 "C-index (95% CI)": fmt(cidx[j], c_lo[j], c_hi[j]),
                 "Adjusted HR (95% CI)": fmt(h[0], h[1], h[2]), "P value": fmt_p(h[3])})
    raw.append({"desc": desc, **base, "threshold_valid": thresholds[j],
                "auroc": auc[j], "auroc_lo": auc_lo[j], "auroc_hi": auc_hi[j],
                "sensitivity": ss[j, 0], "sensitivity_lo": sens_lo[j], "sensitivity_hi": sens_hi[j],
                "specificity": ss[j, 1], "specificity_lo": spec_lo[j], "specificity_hi": spec_hi[j],
                "or_adj": o[0], "or_adj_lo": o[1], "or_adj_hi": o[2], "or_adj_p": o[3], "n_or": o[4],
                "cindex": cidx[j], "cindex_lo": c_lo[j], "cindex_hi": c_hi[j],
                "hr_adj": h[0], "hr_adj_lo": h[1], "hr_adj_hi": h[2], "hr_adj_p": h[3], "n_hr": h[4], "n_hr_events": h[5]})

a_lo, a_hi = ci(auc_caide_b)
k_lo, k_hi = ci(cidx_caide_b)
caide = {"Model": "CAIDE (without education)", "Pretraining domain": "", "Adaptation strategy": ""}
tab2.append({**caide, "AUROC (95% CI)": fmt(auc_caide, a_lo, a_hi)})
tab3.append({**caide, "C-index (95% CI)": fmt(cidx_caide, k_lo, k_hi)})
raw.append({"desc": "caide_noapoe", **caide, "auroc": auc_caide, "auroc_lo": a_lo, "auroc_hi": a_hi,
            "cindex": cidx_caide, "cindex_lo": k_lo, "cindex_hi": k_hi, "n_or": int(caide_t_ok.sum()),
            "n_hr": int(caide_i_ok.sum())})

tab2, tab3 = pd.DataFrame(tab2).fillna(""), pd.DataFrame(tab3).fillna("")
tab2.to_csv(SUMMARY_ROOT / "Table2_detection.csv", index=False, encoding="utf-8-sig")
tab3.to_csv(SUMMARY_ROOT / "Table3_prediction.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(raw).to_csv(SUMMARY_ROOT / "performance_summary_raw.csv", index=False)

print(f"adjusted OR: {ors[0][4]:,} test images with complete covariates | "
      f"adjusted HR: {hrs[0][4]:,} images ({hrs[0][5]:,} incident) with complete covariates")
print(f"[Saved] {SUMMARY_ROOT}/Table2_detection.csv, Table3_prediction.csv, performance_summary_raw.csv")
display(tab2)
display(tab3)

In [ ]:
# Saliency attention-ratio summarizer — Table 4 and Supplementary Tables 2–3
#
# For each of the 21 configurations, reads the per-image regional saliency (Mask/RESULTS_tp.csv written by the main
# loop) and reports the ratio of the mean saliency per pixel in each region to that of the reference region (within
# the field of view, outside the retinal vessels and the optic disc). Ratios are computed per image and averaged over
# images, with 95% CIs from 2,000 bootstrap resamples of images. No GPU is needed.
#
#   detection  = TP images of the test set               (TENSOR_ROOT/<desc>_test)
#   prediction = TP images of the future-incidence cohort (TENSOR_ROOT/<desc>_future)
#
# Outputs (SUMMARY_ROOT):
#   ATTN_summary_detection.csv / ATTN_summary_prediction.csv          — all 21 configurations
#   ATTN_summary_detection_raw.csv / ATTN_summary_prediction_raw.csv  — numeric mean / CI columns
#   Table4_LoRA_detection.csv / Table4_LoRA_prediction.csv             — LoRA configurations only

import numpy as np
import pandas as pd
from pathlib import Path


# -------------------------------------------------------------------------
# Paths — update to match your environment
# -------------------------------------------------------------------------
TENSOR_ROOT  = Path("/path/to/explain_out/tensors")
SUMMARY_ROOT = Path("/path/to/explain_out/summary_tables")
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

N_BOOT    = 2000
BOOT_SEED = 42
REF_COL   = "not_optic_disc_not_both_n"

ARCH_ORDER  = ["retfound", "retfound_dinov2", "openclip", "mae", "dinov2", "dinov3", "dinov3_vith16plus"]
FT_ORDER    = ["linear", "partial", "lora"]
ARCH_PRETTY = {"retfound": "RETFound-MAE", "retfound_dinov2": "RETFound-DINOv2", "openclip": "OpenCLIP",
               "mae": "MAE", "dinov2": "DINOv2", "dinov3": "DINOv3-L", "dinov3_vith16plus": "DINOv3-H+"}
FT_PRETTY   = {"linear": "Linear probing", "partial": "Partial fine-tuning", "lora": "LoRA"}

REGION_TO_SOURCECOL = {
    "Vessels":    "both_n",
    "Arteries":   "arteries_n",
    "Veins":      "veins_n",
    "Optic disc": "optic_disc_n",
    "Temporal":   "optic_disc_temporal_n",
    "Nasal":      "optic_disc_nasal_n",
    "Superior":   "optic_disc_superior_n",
    "Inferior":   "optic_disc_inferior_n",
}


def model_desc(arch, ft):
    if ft == "linear":  return f"{arch}_linear"
    if ft == "partial": return f"{arch}_partial_ft_1"
    return f"{arch}_lora_rank_4_ft_full"


def _safe_ratio(num, den):
    num = pd.to_numeric(num, errors="coerce").to_numpy(float)
    den = pd.to_numeric(den, errors="coerce").to_numpy(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = num / den
    r[~np.isfinite(den) | (den <= 0) | ~np.isfinite(r)] = np.nan
    return r


def bootstrap_ci_mean(x, n_boot=N_BOOT, seed=BOOT_SEED):
    """Mean over images and percentile 95% CI from resampling images with replacement."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan, np.nan, 0
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for s0 in range(0, n_boot, 250):                       # in chunks to limit memory
        b = min(250, n_boot - s0)
        means[s0:s0 + b] = x[rng.integers(0, x.size, size=(b, x.size))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(x.mean()), float(lo), float(hi), int(x.size)


def fmt_ci(m, lo, hi):
    return f"{m:.2f} ({lo:.2f}–{hi:.2f})" if np.isfinite(m) else ""


def collect(tag):
    rows, raws = [], []
    for arch in ARCH_ORDER:
        for ft in FT_ORDER:
            f = TENSOR_ROOT / f"{model_desc(arch, ft)}_{tag}" / "Mask" / "RESULTS_tp.csv"
            if not f.exists():
                continue
            df = pd.read_csv(f)
            ref = df[REF_COL]
            row = {"Model": ARCH_PRETTY[arch], "Adaptation strategy": FT_PRETTY[ft]}
            raw, n_images = dict(row), 0
            for region, col in REGION_TO_SOURCECOL.items():
                m, lo, hi, n = bootstrap_ci_mean(_safe_ratio(df[col], ref))
                row[region] = fmt_ci(m, lo, hi)
                raw.update({f"{region} mean": m, f"{region} lo": lo, f"{region} hi": hi, f"{region} n": n})
                n_images = max(n_images, n)
            row["n images"] = n_images
            rows.append(row)
            raws.append(raw)
    return pd.DataFrame(rows), pd.DataFrame(raws)


for name, tag in (("detection", "test"), ("prediction", "future")):
    tab, raw = collect(tag)
    if not len(tab):
        print(f"[WARN] no RESULTS_tp.csv found under {TENSOR_ROOT} for '{tag}'")
        continue
    tab.to_csv(SUMMARY_ROOT / f"ATTN_summary_{name}.csv", index=False, encoding="utf-8-sig")
    raw.to_csv(SUMMARY_ROOT / f"ATTN_summary_{name}_raw.csv", index=False, encoding="utf-8-sig")
    tab[tab["Adaptation strategy"] == "LoRA"].to_csv(SUMMARY_ROOT / f"Table4_LoRA_{name}.csv",
                                                     index=False, encoding="utf-8-sig")
    print(f"\n=== {name}: saliency ratio relative to the reference region (95% CI, image-level bootstrap) ===")
    display(tab)

print(f"[Saved] {SUMMARY_ROOT}/ATTN_summary_(detection|prediction)[_raw].csv, Table4_LoRA_*.csv")